In [328]:
import os
import polars as pl
import altair as alt

In [329]:
df = pl.read_parquet(os.path.join("data","segments.parquet")).sort(by="date")

In [330]:
df.filter(pl.col("name") == "Risova studanka - od zavory")

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",41326,4626,2024-12-24 23:05:01 CET,2.507177
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",41328,4626,2024-12-25 23:05:55 CET,2.507177
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",41328,4626,2024-12-25 23:35:42 CET,2.507177
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",41336,4626,2024-12-26 23:05:33 CET,2.507177
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",41351,4626,2024-12-27 23:38:33 CET,2.507177
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",43806,5021,2025-06-11 23:38:40 CEST,2.507177
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",43819,5022,2025-06-12 23:22:18 CEST,2.507177
7116683,"""Risova studanka - od zavory""","""Ride""",2612.4,4.6,400.8,279.9,138.5,"[49.228134, 16.500734]","[49.233992, 16.467391]","""Czechia""","""South Moravian Region""","""Brno""",43854,5023,2025-06-13 23:38:27 CEST,2.507177


In [331]:
df.columns

['id',
 'name',
 'activity_type',
 'distance',
 'average_grade',
 'elevation_high',
 'elevation_low',
 'total_elevation_gain',
 'start_latlng',
 'end_latlng',
 'country',
 'state',
 'city',
 'effort_count',
 'athlete_count',
 'date',
 'start_to_finish_distance']

In [332]:
df.filter((pl.col("country") == "Czechia") & (pl.col('distance') > 500) & (pl.col('activity_type') == 'Ride')).select(pl.col('name')).unique().to_series().to_list()

['Chrudim climb',
 'plný kule z kolína',
 'Revnicak Eve',
 'Dářská rašeliniště',
 'Srbsko - Hlásná Třebáň',
 'Ruda -> Gerlovka',
 'Trinec-Tesin',
 'Výstup na Klínovec',
 'Kynšperský ATAK',
 'Výrovka - Modré Sedlo',
 'Starý Kolín - Svatá Kateřina',
 'Špindlerovka - část II. (od Černýho potoka nahoru)',
 'Černý důl konec Hoffmanky',
 'Daskabát flat',
 'Maslovice - Kralupy',
 'Adamaov climb',
 'Kaufland - Popendorf',
 'Final climb ke srubu na Deštnou',
 'Máchovo kritérium - přes bory kolem vrchu Borný',
 'Sptint pod Býčí skálou',
 'Benešovský brdek',
 'Zrucskej smrtak',
 'Klobouky Viadukt',
 'Od křižovatky do Strání po vysílač',
 'Lednice - Bulhary',
 'Mutěnka - z Kyjova po odbočku na hraběcí',
 'Z Lednice po silnici k odbocce na Janohrad',
 'Solan last km',
 'Nová cyklopěšina z Peřiny do HTčka',
 'Cyklostezka sprint směr Děčín',
 'time trial at Svratka',
 'S&Ch20 Masarykův okruh',
 'Papezov - Prazmo',
 'Muglinovská→Hradní lávka (po pravém břehu)',
 'Zlaťák konec - smrt',
 'Třebíč - Staře

In [333]:
def segment(nazev):
    segm = df.filter(pl.col('name') == nazev).group_by_dynamic("date", every="1d").agg(pl.col("effort_count").max())
    segm = segm.with_columns(pl.col("effort_count").diff().alias("daily_difference"))
    segm = segm.with_columns(pl.when(pl.col("daily_difference") < 0).then(None).otherwise(pl.col("daily_difference")).alias("daily_difference"))
    segm = segm.with_columns(pl.col("date").cast(str)).drop_nulls()
    print(segm)
    return alt.Chart(segm, width=800).mark_bar().encode(
        x='date:O',
        y='daily_difference:Q'
    )

In [436]:
segment("BRNĚNSKÁ SA CALOBRA")

shape: (51, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2025-04-22 00:00:00.000000+02:… ┆ 46           ┆ 3                │
│ 2025-04-23 00:00:00.000000+02:… ┆ 67           ┆ 21               │
│ 2025-04-24 00:00:00.000000+02:… ┆ 68           ┆ 1                │
│ 2025-04-25 00:00:00.000000+02:… ┆ 70           ┆ 2                │
│ 2025-04-26 00:00:00.000000+02:… ┆ 73           ┆ 3                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 168          ┆ 2                │
│ 2025-06-12 00:00:00.000000+02:… ┆ 172          ┆ 4                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 173          ┆ 1                │
│ 202

alt.Chart(...)

In [434]:
segment("Junglepark")

shape: (167, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 37930        ┆ 7                │
│ 2024-12-26 00:00:00.000000+01:… ┆ 37944        ┆ 14               │
│ 2024-12-27 00:00:00.000000+01:… ┆ 37953        ┆ 9                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 37925        ┆ 18               │
│ 2024-12-30 00:00:00.000000+01:… ┆ 37932        ┆ 7                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 41447        ┆ 13               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 41458        ┆ 11               │
│ 2025-06-13 00:00:00.000000+02:… ┆ 41473        ┆ 15               │
│ 20

alt.Chart(...)

In [335]:
segment("Revnicak Eve")

shape: (159, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 43467        ┆ 3                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 43469        ┆ 2                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 43471        ┆ 2                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 43478        ┆ 7                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 43481        ┆ 3                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 45980        ┆ 34               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 46006        ┆ 26               │
│ 2025-06-13 00:00:00.000000+02:… ┆ 46037        ┆ 31               │
│ 20

alt.Chart(...)

In [336]:
segment("Sedmička, Lipno {")

shape: (154, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-27 00:00:00.000000+01:… ┆ 12542        ┆ 1                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 12540        ┆ 1                │
│ 2025-01-01 00:00:00.000000+01:… ┆ 12538        ┆ 0                │
│ 2025-01-02 00:00:00.000000+01:… ┆ 12539        ┆ 1                │
│ 2025-01-03 00:00:00.000000+01:… ┆ 12539        ┆ 0                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 13252        ┆ 12               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 13255        ┆ 3                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 13261        ┆ 6                │
│ 20

alt.Chart(...)

In [337]:
segment("Solan last km")

shape: (165, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 12355        ┆ 0                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 12358        ┆ 3                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 12358        ┆ 0                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 12358        ┆ 0                │
│ 2024-12-31 00:00:00.000000+01:… ┆ 12356        ┆ 0                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 12973        ┆ 5                │
│ 2025-06-12 00:00:00.000000+02:… ┆ 12980        ┆ 7                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 12985        ┆ 5                │
│ 20

alt.Chart(...)

In [338]:
segment("Zweite hup u ZOO")

shape: (167, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 233387       ┆ 37               │
│ 2024-12-26 00:00:00.000000+01:… ┆ 233414       ┆ 27               │
│ 2024-12-27 00:00:00.000000+01:… ┆ 233427       ┆ 13               │
│ 2024-12-28 00:00:00.000000+01:… ┆ 233437       ┆ 10               │
│ 2024-12-29 00:00:00.000000+01:… ┆ 233460       ┆ 23               │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 249785       ┆ 193              │
│ 2025-06-12 00:00:00.000000+02:… ┆ 249968       ┆ 183              │
│ 2025-06-13 00:00:00.000000+02:… ┆ 250226       ┆ 258              │
│ 20

alt.Chart(...)

In [339]:
segment("TJ Sokol okruh")

shape: (167, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 72623        ┆ 32               │
│ 2024-12-27 00:00:00.000000+01:… ┆ 72623        ┆ 0                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 72635        ┆ 12               │
│ 2024-12-29 00:00:00.000000+01:… ┆ 72635        ┆ 0                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 72669        ┆ 34               │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 84222        ┆ 69               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 84338        ┆ 116              │
│ 2025-06-13 00:00:00.000000+02:… ┆ 84489        ┆ 151              │
│ 20

alt.Chart(...)

In [340]:
segment('400 metrů, 44 120 diváků')

shape: (169, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 4461         ┆ 0                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 4461         ┆ 0                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 4461         ┆ 0                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 4461         ┆ 0                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 4461         ┆ 0                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 4655         ┆ 0                │
│ 2025-06-12 00:00:00.000000+02:… ┆ 4655         ┆ 0                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 4655         ┆ 0                │
│ 20

alt.Chart(...)

In [341]:
def weekly(segment):
    return df.filter(
        pl.col('name') == segment
    ).group_by_dynamic("date", every="1d").agg(
        pl.col("effort_count").max()
    ).with_columns(
        pl.col("effort_count").diff().alias("daily_difference")
    ).with_columns(
        pl.when(pl.col("daily_difference") < 0).then(None).otherwise(pl.col("daily_difference")).alias("daily_difference")
    ).drop_nulls(
    ).with_columns(
        pl.col("date").dt.weekday().alias("day")
    ).group_by("day").agg(pl.col("daily_difference").median()).sort(by="day")

In [342]:
def graf(tyden):
    return alt.Chart(tyden).mark_bar().encode(alt.X("day:N"), alt.Y("daily_difference:Q"))

In [343]:
graf(weekly("Track Domažlice"))

alt.Chart(...)

In [344]:
graf(weekly("Velká Deštná climb "))

alt.Chart(...)

In [345]:
graf(weekly("Židlochovice -> Olympia"))

alt.Chart(...)

In [346]:
graf(weekly("Od posledního tunelu do Bílovic"))

alt.Chart(...)

In [347]:
segment("Opičí Časovka, Podolská vodárna ")

shape: (0, 3)
┌──────┬──────────────┬──────────────────┐
│ date ┆ effort_count ┆ daily_difference │
│ ---  ┆ ---          ┆ ---              │
│ str  ┆ i64          ┆ i64              │
╞══════╪══════════════╪══════════════════╡
└──────┴──────────────┴──────────────────┘


alt.Chart(...)

In [348]:
graf(weekly("Opičí Časovka, Podolská vodárna"))

alt.Chart(...)

In [349]:
segment("Vítkov up sprint ")

shape: (0, 3)
┌──────┬──────────────┬──────────────────┐
│ date ┆ effort_count ┆ daily_difference │
│ ---  ┆ ---          ┆ ---              │
│ str  ┆ i64          ┆ i64              │
╞══════╪══════════════╪══════════════════╡
└──────┴──────────────┴──────────────────┘


alt.Chart(...)

In [350]:
df.filter(pl.col("name").str.contains("Vítkov up sprint"))

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86417,8588,2024-12-27 23:22:03 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86426,8589,2024-12-28 23:05:37 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86438,8592,2024-12-29 23:38:26 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86458,8598,2024-12-30 23:21:42 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86473,8600,2024-12-31 23:22:02 CET,0.117786
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",91542,9498,2025-06-11 23:05:20 CEST,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",91578,9504,2025-06-12 23:21:58 CEST,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",91598,9504,2025-06-13 23:05:27 CEST,0.117786


In [351]:
graf(weekly("Vítkov up sprint"))

alt.Chart(...)

In [352]:
segment("Risova studanka - od zavory")

shape: (168, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 41328        ┆ 2                │
│ 2024-12-26 00:00:00.000000+01:… ┆ 41336        ┆ 8                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 41351        ┆ 15               │
│ 2024-12-28 00:00:00.000000+01:… ┆ 41360        ┆ 9                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 41365        ┆ 5                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 43806        ┆ 30               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 43819        ┆ 13               │
│ 2025-06-13 00:00:00.000000+02:… ┆ 43854        ┆ 35               │
│ 20

alt.Chart(...)

In [353]:
segment("Sedýlko - Praděd climb")

shape: (141, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 31962        ┆ 0                │
│ 2024-12-26 00:00:00.000000+01:… ┆ 31964        ┆ 2                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 31969        ┆ 5                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 31976        ┆ 7                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 31977        ┆ 1                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 32718        ┆ 2                │
│ 2025-06-12 00:00:00.000000+02:… ┆ 32735        ┆ 17               │
│ 2025-06-13 00:00:00.000000+02:… ┆ 32765        ┆ 30               │
│ 20

alt.Chart(...)

In [354]:
segment("Malchor - Lysá Hora")

shape: (169, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 19801        ┆ 47               │
│ 2024-12-27 00:00:00.000000+01:… ┆ 19835        ┆ 34               │
│ 2024-12-28 00:00:00.000000+01:… ┆ 19870        ┆ 35               │
│ 2024-12-29 00:00:00.000000+01:… ┆ 19910        ┆ 40               │
│ 2024-12-30 00:00:00.000000+01:… ┆ 19952        ┆ 42               │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 21822        ┆ 11               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 21834        ┆ 12               │
│ 2025-06-13 00:00:00.000000+02:… ┆ 21847        ┆ 13               │
│ 20

alt.Chart(...)

In [355]:
graf(weekly("Tyršák čtyřstofka"))

alt.Chart(...)

In [356]:
graf(weekly('Juliska 400 m'))

alt.Chart(...)

In [357]:
graf(weekly('Karlův most ONLY'))

alt.Chart(...)

In [358]:
segment("Ořešník")

shape: (157, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 1105         ┆ 1                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 1105         ┆ 0                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 1106         ┆ 1                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 1105         ┆ 1                │
│ 2024-12-31 00:00:00.000000+01:… ┆ 1106         ┆ 1                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 1188         ┆ 0                │
│ 2025-06-12 00:00:00.000000+02:… ┆ 1189         ┆ 1                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 1189         ┆ 0                │
│ 20

alt.Chart(...)

In [359]:
df = pl.read_parquet("data_raw/segments/*.parquet")

df = df.with_columns(
    (pl.col("date").cast(pl.Int64) * 1000000).cast(pl.Datetime(time_zone='CET'))
)
df = df.filter(pl.col('date').dt.hour() > 22)
df = df.sort(by="date")

In [360]:
df.columns

['id',
 'resource_state',
 'name',
 'activity_type',
 'distance',
 'average_grade',
 'maximum_grade',
 'elevation_high',
 'elevation_low',
 'start_latlng',
 'end_latlng',
 'elevation_profile',
 'elevation_profiles',
 'climb_category',
 'city',
 'state',
 'country',
 'private',
 'hazardous',
 'starred',
 'created_at',
 'updated_at',
 'total_elevation_gain',
 'map',
 'effort_count',
 'athlete_count',
 'star_count',
 'athlete_segment_stats',
 'xoms',
 'local_legend',
 'date']

In [361]:
df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())

C:\Users\micha\AppData\Local\Temp\ipykernel_32152\3197359101.py:1: DeprecationWarning: the argument `by` for `DataFrame.group_by_dynamic` is deprecated. It was renamed to `group_by` in version 0.20.14.
  df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())


name,date,effort_count
str,"datetime[μs, CET]",i64
"""400m""",2024-12-26 00:00:00 CET,33521
"""400m""",2024-12-27 00:00:00 CET,33521
"""400m""",2024-12-28 00:00:00 CET,33518
"""400m""",2024-12-29 00:00:00 CET,33518
"""400m""",2024-12-30 00:00:00 CET,33518
…,…,…
"""Szlak czerwony, Schronisko Nad…",2025-06-11 00:00:00 CEST,9541
"""Szlak czerwony, Schronisko Nad…",2025-06-12 00:00:00 CEST,9558
"""Szlak czerwony, Schronisko Nad…",2025-06-13 00:00:00 CEST,9565


In [362]:
df

id,resource_state,name,activity_type,distance,average_grade,maximum_grade,elevation_high,elevation_low,start_latlng,end_latlng,elevation_profile,elevation_profiles,climb_category,city,state,country,private,hazardous,starred,created_at,updated_at,total_elevation_gain,map,effort_count,athlete_count,star_count,athlete_segment_stats,xoms,local_legend,date
i64,i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,struct[2],i64,str,str,str,bool,bool,bool,str,str,f64,struct[3],i64,i64,i64,struct[6],struct[4],struct[7],"datetime[μs, CET]"
17521739,3,"""Dlouhe Strane od Loucnej""","""Ride""",12619.0,6.7,38.0,1333.5,492.8,"[50.069963, 17.091625]","[50.077466, 17.163214]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/OC6CGKN3U7SBXOGZOJBR6TZPDQSIOLK6LNKDEA5BVKFNGWWVWBSRKLL4SPOAGEECOMJUFTIJQ2XV5UVHNG75ABI="",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/QJQO5N7WHHPKXZIWHZIHJXGVPSY5NMSILNWETIH4KA2SFBG3ZNRVLHMDRAPHTJ6FZMKVJTDWBGKEVYGV7OWJOYI=""}",5,"""Loučná nad Desnou""","""Olomoucký kraj""","""Czechia""",false,false,true,"""2018-04-29T19:08:58Z""","""2021-05-15T02:20:58Z""",891.8,"{""s17521739"",""ghrpHseigBa@YEIU_AC[J}@Pm@Jm@BcAKRIA_@BU?]Ke@E_@O{@u@}@aAa@YWKc@[K?[So@cAQOGAEG_AyAGAUYa@y@Y[e@m@o@kBa@w@KYMGcA}BCOUi@YcBQk@]g@Uo@]u@CEGUOQg@w@_AaCEEKUa@e@[}@KOIWQ[q@w@c@o@WSMES]a@a@U[u@q@KUOc@e@q@s@m@_@_@CGQK]_@WQMUUs@OIC?QOCIEA]YYOGEM_@_@US[E[OWESe@{@a@o@OKo@q@[QY_@QMw@e@GMi@u@GOEa@Cg@?IFYFEl@Dn@JDAlABdANd@CFD`@KHB\E^FPAX@XGVBt@ERCb@?bAHj@@XDTCb@Bt@KFKVO\_@f@w@VW~@u@P]LMRc@ZgAP}@b@gAJg@Pi@Fg@Z{@\gBJ[A_@@IDYJQHM\IVSJ@r@MBETENIt@c@h@GTSFOJAv@HLCJIHM?UBKRU@SJQDsABSHkA@FPNh@NFDLV@TCJFPFDDMJh@b@Vl@`ANZ@LFP@LNd@JID@JN@F@l@Cb@@d@Fl@DHJN\LIe@?a@De@Vw@@KRu@Fs@Lo@Do@AUDa@Aa@BU?y@OeBFc@Ay@NgACi@@WF[FyAJIJ_@Hs@TkADs@Ai@O_AGyAGm@@UESCg@]aB]}@UWGY]m@Mg@Sg@E[Us@EEU?SP]PMLKBMJMLKVw@r@Od@MRERORSPMNQN]No@j@WPILUFG?WLOPE@YRGJi@Z[J]Zc@VOLk@Lk@XM@o@NY?MEw@@KAMDa@?KCE@SIMAIIKAI@KEkAUE?[GIEKMOk@Sg@Ie@ASGk@B]Eg@Fe@Ms@BWAq@Bw@A}@BEOqAAk@G_@Ag@E_@?k@MuACqA?g@C[O}AMk@Ge@[iBM]Eo@YkB?iACUB{@CO?MBq@F[EOA_@DQ@q@Hg@C_@N}BA]CQBGAUB_@P}A@_@F}@Ha@Cw@KeAWMCGMCaAz@i@X[V_@FSNE?SBYLEAM@YLg@V[Ze@RONa@Tg@JQI[CECGAOSKWSGO]iAuAWi@OMISKGS[CGOOGQE?[i@]U}@gAQMk@{@MESUWOIK]Wm@SIUECOC]W]KEGYM_@OUYKGKMWKQMKMEASMKKEMYWYQQQIO[YQIIKE?CCKBUEK?WJMC{@Ng@KSMOQSI_@w@m@g@AKa@UW?UMQGsAcAQSKUi@q@u@}A[sAOi@Km@?UFQTa@PUHOLGRUVKZWd@GX@ZGX?l@DJFLANFTABED@DA|@YJALIL@PEPBFAL@`@IL@ZALEDAD@JELBPFNK?IIc@Yo@Og@E[Ki@IMAWOo@A_@@]Co@G_@?_@I]MeBOw@C_@Qi@Ea@a@gBEk@@WTk@HIZOTGD@LCP@vAn@jBh@d@T~@VRAREj@A`@S^GZY\Kx@y@h@a@TGXShAc@tBaAbCoA`BaAv@]pBmAxFuDhAk@pBkAdAu@Z[z@sAh@m@LWDSGKEASOYEq@_@M@UOSIIISEUMCIOIGGI@KO]OGKMK_@s@EC?IQc@Ic@SUKWm@Ko@SG@SEYAOEWCGEe@CSII@MCSBGFS@IFq@AED]JEEKA_@Fe@EYEWMK?OEYIUMc@SE@USCGQKUw@Mk@Gm@Bq@Ja@D]ZaAHQx@{Cn@sB?GDGVgA^kAP[H]x@}A@IJO@IP[Ja@NSBSX]Po@JS@MJMHUTa@@UNS@QP}@Fs@FOLq@@WJe@Fw@Pu@?GHi@Dg@?a@Hw@Ia@@wCKiCQgAWcAGs@My@Co@F]Ty@Ti@\i@BIx@k@NEBELEBED?TKRQb@G^BZ@TJFCRBPHXBZRPDRVVNDHb@LRTh@VNTL?FJH@HLf@Tz@d@DHHHF@BDD?LLLBRVD?HHF@NPF`@KVEBc@MKIGCYSYGe@EG@g@K]CGCG@u@MgA?UDc@PGJIDY`@M\QVK\MdACd@Bb@AbCd@zDJf@@LJh@LJN?JCJ_@GeB?c@Ca@FiAJy@f@aARUP[ZSRWDCPB"",3}",308,214,7,"{null,null,null,null,null,0}","{""43:55"",""1:00:24"",""43:55"",{""strava://segments/17521739/leaderboard"",""overall"",""All-Time""}}","{13919744,""Kamil Filipek"",""https://dgalywyr863hv.cloudfront.net/pictures/athletes/13919744/6285929/3/large.jpg"",""1 effort in the last 90 days"",""1"",{""1 effort"",null},""strava://segments/17521739/local_legend?categories%5B%5D=overall""}",2024-12-24 23:02:03 CET
8598488,3,"""Lysa hora - jizni sjezdovka""","""Run""",298.4,21.3,36.0,1298.2,1234.6,"[49.544042, 18.452326]","[49.545601, 18.449198]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/5J5C33G27OKLY36XOMNHJJAEARTJAFC73RAKGNINHOEVXYUAVONOBEMXWTMUK7M2W3TVJLTWSOJPY24KN5SUUSY="",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/5CF2ETIB6LKDHWIDTGUS6ARGAUMFXUFVBHDUIUYMZLUO6Q47MANJNIOJK7U3TMDMA5UO6HNCVGGYAIALJSCONNA=""}",0,"""Ostravice""","""Moravian-Silesian

In [363]:
segment('Lužánky rovinka')

shape: (168, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 179488       ┆ 40               │
│ 2024-12-26 00:00:00.000000+01:… ┆ 179527       ┆ 39               │
│ 2024-12-27 00:00:00.000000+01:… ┆ 179560       ┆ 33               │
│ 2024-12-28 00:00:00.000000+01:… ┆ 179584       ┆ 24               │
│ 2024-12-29 00:00:00.000000+01:… ┆ 179626       ┆ 42               │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 195805       ┆ 121              │
│ 2025-06-12 00:00:00.000000+02:… ┆ 195927       ┆ 122              │
│ 2025-06-13 00:00:00.000000+02:… ┆ 196015       ┆ 88               │
│ 20

alt.Chart(...)

In [364]:
segment('time trial at Svratka')

shape: (163, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 141661       ┆ 24               │
│ 2024-12-26 00:00:00.000000+01:… ┆ 141680       ┆ 19               │
│ 2024-12-27 00:00:00.000000+01:… ┆ 141706       ┆ 26               │
│ 2024-12-28 00:00:00.000000+01:… ┆ 141721       ┆ 15               │
│ 2024-12-29 00:00:00.000000+01:… ┆ 141735       ┆ 14               │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 149597       ┆ 125              │
│ 2025-06-12 00:00:00.000000+02:… ┆ 149717       ┆ 120              │
│ 2025-06-13 00:00:00.000000+02:… ┆ 149833       ┆ 116              │
│ 20

alt.Chart(...)

In [365]:
segment('Od posledního tunelu do Bílovic')

shape: (161, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 124240       ┆ 14               │
│ 2024-12-27 00:00:00.000000+01:… ┆ 124314       ┆ 74               │
│ 2024-12-29 00:00:00.000000+01:… ┆ 124308       ┆ 18               │
│ 2024-12-30 00:00:00.000000+01:… ┆ 124325       ┆ 17               │
│ 2024-12-31 00:00:00.000000+01:… ┆ 124334       ┆ 9                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 133685       ┆ 143              │
│ 2025-06-12 00:00:00.000000+02:… ┆ 133792       ┆ 107              │
│ 2025-06-13 00:00:00.000000+02:… ┆ 133901       ┆ 109              │
│ 20

alt.Chart(...)

In [366]:
segment('Lužánky - Plochá dráha')

shape: (156, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-27 00:00:00.000000+01:… ┆ 32752        ┆ 2                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 32746        ┆ 0                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 32763        ┆ 17               │
│ 2024-12-31 00:00:00.000000+01:… ┆ 32763        ┆ 0                │
│ 2025-01-02 00:00:00.000000+01:… ┆ 32762        ┆ 5                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 35171        ┆ 18               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 35226        ┆ 55               │
│ 2025-06-13 00:00:00.000000+02:… ┆ 35252        ┆ 26               │
│ 20

alt.Chart(...)

In [367]:
segment('Risova studanka - od zavory')

shape: (168, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 41328        ┆ 2                │
│ 2024-12-26 00:00:00.000000+01:… ┆ 41336        ┆ 8                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 41351        ┆ 15               │
│ 2024-12-28 00:00:00.000000+01:… ┆ 41360        ┆ 9                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 41365        ┆ 5                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 43806        ┆ 30               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 43819        ┆ 13               │
│ 2025-06-13 00:00:00.000000+02:… ┆ 43854        ┆ 35               │
│ 20

alt.Chart(...)

In [368]:
segment('Brno Circuit')

shape: (167, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-28 00:00:00.000000+01:… ┆ 880          ┆ 0                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 881          ┆ 1                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 881          ┆ 0                │
│ 2024-12-31 00:00:00.000000+01:… ┆ 881          ┆ 0                │
│ 2025-01-01 00:00:00.000000+01:… ┆ 881          ┆ 0                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 878          ┆ 0                │
│ 2025-06-12 00:00:00.000000+02:… ┆ 878          ┆ 0                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 878          ┆ 0                │
│ 20

alt.Chart(...)

In [369]:
df.filter(pl.col('country').str.contains('Cze')).filter(pl.col('activity_type') == 'Run').group_by("name").agg(pl.col('effort_count').cast(int).max()).sort(by='effort_count',descending=True)

name,effort_count
str,i64
"""Stadion Na Děkance""",230441
"""Lužánky rovinka""",196161
"""Šlechtovka - finální rovinka""",172622
"""VUT horní dráha 400m""",126845
"""Strahovská dráha""",110129
…,…
"""Podél tratě do Úval""",345
"""Valy""",311
"""Autodrom Most""",270


In [370]:
set(pl.Series(df.filter((pl.col('activity_type') == 'Run') & (pl.col('city') == 'Ostrava')).select("name")).to_list())

{'Hradní lávka => Sýkorův most',
 'Opavska down 1/2',
 'Park 1',
 'Železniční most => Jez Vítkovice'}

In [371]:
segment('Pražačka 320')

shape: (153, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-27 00:00:00.000000+01:… ┆ 74840        ┆ 23               │
│ 2024-12-29 00:00:00.000000+01:… ┆ 74919        ┆ 101              │
│ 2024-12-30 00:00:00.000000+01:… ┆ 74938        ┆ 19               │
│ 2024-12-31 00:00:00.000000+01:… ┆ 74961        ┆ 23               │
│ 2025-01-01 00:00:00.000000+01:… ┆ 75036        ┆ 75               │
│ …                               ┆ …            ┆ …                │
│ 2025-06-10 00:00:00.000000+02:… ┆ 83242        ┆ 108              │
│ 2025-06-11 00:00:00.000000+02:… ┆ 83641        ┆ 399              │
│ 2025-06-12 00:00:00.000000+02:… ┆ 84064        ┆ 423              │
│ 20

alt.Chart(...)

In [372]:
segment('Park 1')

shape: (169, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000+01:… ┆ 13566        ┆ 6                │
│ 2024-12-26 00:00:00.000000+01:… ┆ 13571        ┆ 5                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 13571        ┆ 0                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 13573        ┆ 2                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 13576        ┆ 3                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 15178        ┆ 44               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 15186        ┆ 8                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 15193        ┆ 7                │
│ 20

alt.Chart(...)

In [373]:
segment('Hradní lávka => Sýkorův most')

shape: (168, 3)
┌─────────────────────────────────┬──────────────┬──────────────────┐
│ date                            ┆ effort_count ┆ daily_difference │
│ ---                             ┆ ---          ┆ ---              │
│ str                             ┆ i64          ┆ i64              │
╞═════════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000+01:… ┆ 13983        ┆ 6                │
│ 2024-12-27 00:00:00.000000+01:… ┆ 13985        ┆ 2                │
│ 2024-12-28 00:00:00.000000+01:… ┆ 13987        ┆ 2                │
│ 2024-12-29 00:00:00.000000+01:… ┆ 13992        ┆ 5                │
│ 2024-12-30 00:00:00.000000+01:… ┆ 13998        ┆ 6                │
│ …                               ┆ …            ┆ …                │
│ 2025-06-11 00:00:00.000000+02:… ┆ 15530        ┆ 11               │
│ 2025-06-12 00:00:00.000000+02:… ┆ 15538        ┆ 8                │
│ 2025-06-13 00:00:00.000000+02:… ┆ 15553        ┆ 15               │
│ 20

alt.Chart(...)

In [374]:
df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())

C:\Users\micha\AppData\Local\Temp\ipykernel_32152\3197359101.py:1: DeprecationWarning: the argument `by` for `DataFrame.group_by_dynamic` is deprecated. It was renamed to `group_by` in version 0.20.14.
  df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())


name,date,effort_count
str,"datetime[μs, CET]",i64
"""400m""",2024-12-26 00:00:00 CET,33521
"""400m""",2024-12-27 00:00:00 CET,33521
"""400m""",2024-12-28 00:00:00 CET,33518
"""400m""",2024-12-29 00:00:00 CET,33518
"""400m""",2024-12-30 00:00:00 CET,33518
…,…,…
"""Szlak czerwony, Schronisko Nad…",2025-06-11 00:00:00 CEST,9541
"""Szlak czerwony, Schronisko Nad…",2025-06-12 00:00:00 CEST,9558
"""Szlak czerwony, Schronisko Nad…",2025-06-13 00:00:00 CEST,9565
